# 01 - Data Loading, Cleaning, and Preprocessing

Used car price prediction, step 1 of the project. Data Science course task: predict the selling price of a used car and deploy the model as a Streamlit app.

Each step of the project has its own notebook:

- **01 - data loading, exploration, cleaning, and preprocessing (this notebook)**
- 02 - exploratory data analysis

This notebook reads the raw CSV and saves a cleaned file (`cardekho_cleaned.csv`) that the next notebooks start from.

Dataset: CarDekho used car data from Kaggle (Indian market, prices in rupees).

## 1. Setup

In [1]:
# Import the libraries used in this notebook
import pandas as pd
import numpy as np

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Show every column when printing tables
pd.set_option("display.max_columns", None)

# Print the library versions. We will pin these in requirements.txt later, so the model
# saved here can be loaded on the local PC without version problems.
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)

pandas: 2.2.3
numpy: 2.4.6
scikit-learn: 1.6.1


## 2. Load the data

We load the raw CSV with Pandas. The raw file is never edited. All cleaning happens in this notebook and the result is saved as a separate file.

In [2]:
# Load the raw CSV with Pandas.
# First try to read it straight from the GitHub repo. If the repo is private or the
# link fails, fall back to uploading the CSV file by hand (a file picker will appear).
DATA_URL = "https://raw.githubusercontent.com/MirajAdhikari/used-car-price-prediction/main/data/raw/cardekho_dataset.csv"

try:
    df = pd.read_csv(DATA_URL)
    print("Loaded the data from GitHub.")
except Exception:
    from google.colab import files
    uploaded = files.upload()                       # choose cardekho_dataset.csv
    df = pd.read_csv(next(iter(uploaded)))
    print("Loaded the data from the uploaded file.")

print("Shape (rows, columns):", df.shape)


Loaded the data from GitHub.
Shape (rows, columns): (15411, 14)


## 3. Explore the raw data

First look: the first rows, the column types, and basic statistics.

In [3]:
# First 5 rows, to see what each column looks like
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [4]:
# Column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB


In [5]:
# Summary statistics for the numeric columns (look at min and max for odd values)
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,15411.0,9811.857699,5643.418542,0.0,4906.5,9872.00,14668.5,19543.00
vehicle_age,15411.0,6.036338,3.013291,0.0,4.0,6.00,8.0,29.00
km_driven,15411.0,55616.480631,51618.548422,100.0,30000.0,50000.00,70000.0,3800000.00
mileage,15411.0,19.701151,4.171265,4.0,17.0,19.67,22.7,33.54
engine,15411.0,1486.057751,521.106696,793.0,1197.0,1248.00,1582.0,6592.00
max_power,15411.0,100.588254,42.972979,38.4,74.0,88.50,117.3,626.00
seats,15411.0,5.325482,0.807628,0.0,5.0,5.00,5.0,9.00
selling_price,15411.0,774971.116410,894128.363263,40000.0,385000.0,556000.00,825000.0,39500000.00


### Missing values and duplicates

In [6]:
# Count missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

# Count rows that are exact duplicates. We ignore the first column ("Unnamed: 0"),
# because it is only a row index and makes every row look different.
duplicates_before = df.drop(columns=["Unnamed: 0"]).duplicated().sum()
print("\nDuplicate rows (ignoring the index column):", duplicates_before)

Missing values per column:
Unnamed: 0           0
car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

Duplicate rows (ignoring the index column): 167


### Categorical columns

In [7]:
# Look at the categories in each text column.
# We are checking for spelling or capitalization differences (e.g. "ISUZU" vs "Isuzu").
for col in ["brand", "seller_type", "fuel_type", "transmission_type"]:
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    print(df[col].value_counts().to_string())
    print()

--- brand (32 unique) ---
brand
Maruti           4992
Hyundai          2982
Honda            1485
Mahindra         1011
Toyota            793
Ford              790
Volkswagen        620
Renault           536
BMW               439
Tata              430
Mercedes-Benz     337
Skoda             334
Audi              192
Datsun            170
Jaguar             59
Land Rover         51
Jeep               41
Kia                32
Porsche            21
Volvo              20
MG                 19
Mini               17
Nissan             11
Lexus              10
Isuzu               8
Bentley             3
Maserati            2
ISUZU               2
Ferrari             1
Mercedes-AMG        1
Rolls-Royce         1
Force               1

--- seller_type (3 unique) ---
seller_type
Dealer              9539
Individual          5699
Trustmark Dealer     173

--- fuel_type (5 unique) ---
fuel_type
Petrol      7643
Diesel      7419
CNG          301
LPG           44
Electric       4

--- transmission_ty

### Suspicious values

In [8]:
# Check the columns where we saw odd minimum or maximum values in describe()
print("Cars with 0 seats:", (df["seats"] == 0).sum())
print("Cars with more than 1,000,000 km driven:", (df["km_driven"] > 1_000_000).sum())

print("\nHighest km_driven:")
print(df.sort_values("km_driven", ascending=False)
        .head(5)[["car_name", "vehicle_age", "km_driven", "selling_price"]]
        .to_string())

print("\nHighest prices (luxury cars):")
print(df.sort_values("selling_price", ascending=False)
        .head(5)[["car_name", "vehicle_age", "km_driven", "selling_price"]]
        .to_string())

Cars with 0 seats: 2
Cars with more than 1,000,000 km driven: 2

Highest km_driven:
                    car_name  vehicle_age  km_driven  selling_price
15409        Mahindra XUV500            5    3800000        1225000
11903  Mercedes-Benz E-Class            9    1325000        1350000
8772           Toyota Innova            6     950000         950000
4532          Renault Duster            4     850000         850000
6712        Volkswagen Vento            9     830000         380000

Highest prices (luxury cars):
                    car_name  vehicle_age  km_driven  selling_price
3799       Ferrari GTC4Lusso            2       3800       39500000
10969      Rolls-Royce Ghost            4       5000       24200000
1172     Bentley Continental            9       9000       14500000
9722   Mercedes-Benz S-Class            3       4000       13000000
9364         Porsche Cayenne            4      24000       11100000


### What we found

- No missing values.
- Some exact duplicate rows.
- A few impossible values: 0 seats, and extreme `km_driven` values that are not realistic for the age of the car.
- Inconsistent brand spelling (e.g. "ISUZU" and "Isuzu").
- Some very expensive luxury cars. These are real cars, not errors, so we keep them.
- `Unnamed: 0` is just a row index, and `car_name` is brand + model written together, so neither adds information.

## 4. Clean the data

Each step below is logged, so we can show how many rows were removed and why.

In [9]:
# Work on a copy, so the raw data in `df` stays untouched
clean = df.copy()

# A log that records the row count after each cleaning step
cleaning_log = [("Raw data", len(clean))]

**Step 1: drop columns that add no information.** `Unnamed: 0` is a row index, and `car_name` is just brand + model.

In [10]:
# Confirm that car_name really is brand + model for every row before dropping it
print("car_name equals brand + model in every row:",
      (clean["car_name"] == clean["brand"] + " " + clean["model"]).all())

clean = clean.drop(columns=["Unnamed: 0", "car_name"])
print("Columns now:", list(clean.columns))

car_name equals brand + model in every row: True
Columns now: ['brand', 'model', 'vehicle_age', 'km_driven', 'seller_type', 'fuel_type', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price']


**Step 2: make text labels consistent.** If the same label is written with different capitalization, we use the most common spelling.

In [11]:
text_cols = ["brand", "model", "seller_type", "fuel_type", "transmission_type"]
brands_before = clean["brand"].nunique()

for col in text_cols:
    # Remove spaces at the start and end of each label
    clean[col] = clean[col].str.strip()
    # For each label (ignoring capitalization), find the spelling used most often
    most_common = clean.groupby(clean[col].str.lower())[col].agg(lambda s: s.value_counts().idxmax())
    # Replace every spelling with that most common one
    clean[col] = clean[col].str.lower().map(most_common)

print("Unique brands before:", brands_before, "| after:", clean["brand"].nunique())
print("Isuzu rows:", (clean["brand"] == "Isuzu").sum())

Unique brands before: 32 | after: 31
Isuzu rows: 10


**Step 3: remove duplicate rows.** Rows that match in every column (same car, same details, same price) are treated as duplicate listings.

In [12]:
rows_before = len(clean)
clean = clean.drop_duplicates()
print("Duplicate rows removed:", rows_before - len(clean))
cleaning_log.append(("After removing duplicates", len(clean)))

Duplicate rows removed: 167


**Step 4: remove impossible values.** A car cannot have 0 seats.

In [13]:
rows_before = len(clean)
clean = clean[clean["seats"] > 0]
print("Rows with 0 seats removed:", rows_before - len(clean))
cleaning_log.append(("After removing 0-seat rows", len(clean)))

Rows with 0 seats removed: 2


**Step 5: remove unrealistic kilometers.** A car with more than 1,000,000 km driven is almost certainly a data entry error. For example, a 5-year-old car with 3,800,000 km would need over 700,000 km per year.

We do not remove the luxury cars. Their prices are real, and the price skew will be handled during modeling (we compare raw price against log price in Part 5).

In [14]:
rows_before = len(clean)
clean = clean[clean["km_driven"] <= 1_000_000]
print("Rows with more than 1,000,000 km removed:", rows_before - len(clean))
cleaning_log.append(("After removing extreme km_driven", len(clean)))

# Reset the row numbers after removing rows
clean = clean.reset_index(drop=True)

Rows with more than 1,000,000 km removed: 2


In [15]:
# Cleaning summary: row count after each step
log_table = pd.DataFrame(cleaning_log, columns=["Step", "Rows"])
log_table["Removed"] = log_table["Rows"].diff().abs().fillna(0).astype(int)
log_table

,Step,Rows,Removed
0,Raw data,15411,0
1,After removing duplicates,15244,167
2,After removing 0-seat rows,15242,2
3,After removing extreme km_driven,15240,2


## 5. Create a new feature

`km_per_year` is the average distance driven per year of age. Two cars with the same total kilometers can be very different: 60,000 km in 2 years means heavy use, while 60,000 km in 10 years means light use. This feature captures that.

A brand new car (age 0) would cause a division by zero, so we treat age 0 as 1 year.

In [16]:
clean["km_per_year"] = clean["km_driven"] / clean["vehicle_age"].clip(lower=1)

# Check the new feature
clean["km_per_year"].describe()

,km_per_year
count,15240.000000
mean,9800.151768
std,7323.313834
min,100.000000
25%,5833.333333
50%,8428.571429
75%,12166.666667
max,295000.000000


## 6. Preprocessing: encoding and scaling

Machine learning models need numbers, so we convert the text columns to numbers (**one-hot encoding**) and put the numeric columns on the same scale (**standardization**).

Important: encoders and scalers must learn from the **training data only**. If they learn from the full dataset, information from the test set leaks into training and the results look better than they really are. So we:

1. Split the data first.
2. Build a `ColumnTransformer` that holds both steps.
3. Fit it on the training part only.

In the modeling notebook the same `ColumnTransformer` will be placed inside a scikit-learn `Pipeline` together with the model. The Streamlit app will then use exactly the same preprocessing as training.

In [17]:
# Target (what we predict) and the two groups of input features
target = "selling_price"

numeric_features = ["vehicle_age", "km_driven", "mileage", "engine",
                    "max_power", "seats", "km_per_year"]
categorical_features = ["brand", "model", "seller_type", "fuel_type", "transmission_type"]

X = clean[numeric_features + categorical_features]
y = clean[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (15240, 12)
y shape: (15240,)


In [18]:
# Split into 80% training and 20% testing.
# random_state makes the split the same every time (every notebook reuses 42).
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))

Training rows: 12192 | Test rows: 3048


In [19]:
# The preprocessor:
#  - numeric columns are standardized (mean 0, standard deviation 1)
#  - text columns are one-hot encoded; handle_unknown="ignore" means a category that
#    never appeared in training (for example a rare brand) does not crash the app
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# Fit on the TRAINING data only, then transform both parts
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Training matrix shape:", X_train_encoded.shape)
print("Test matrix shape:", X_test_encoded.shape)
print("Number of encoded feature columns:", len(preprocessor.get_feature_names_out()))

Training matrix shape: (12192, 164)
Test matrix shape: (3048, 164)
Number of encoded feature columns: 164


In [20]:
# Show a few of the new column names, to see what the encoding produced
names = preprocessor.get_feature_names_out()
print(list(names[:10]))
print("...")
print(list(names[-5:]))

['num__vehicle_age', 'num__km_driven', 'num__mileage', 'num__engine', 'num__max_power', 'num__seats', 'num__km_per_year', 'cat__brand_Audi', 'cat__brand_BMW', 'cat__brand_Bentley']
...
['cat__fuel_type_Electric', 'cat__fuel_type_LPG', 'cat__fuel_type_Petrol', 'cat__transmission_type_Automatic', 'cat__transmission_type_Manual']


## 7. Save the cleaned data

The cleaned dataset is saved as a separate file. The raw file stays unchanged.

In [21]:
# Save the cleaned data as a CSV file
clean.to_csv("cardekho_cleaned.csv", index=False)
print("Saved cleaned data:", clean.shape)

# In Colab, download the file to your PC. Put it in data/processed/ in the repo.
try:
    from google.colab import files
    files.download("cardekho_cleaned.csv")
except ImportError:
    print("Not running in Colab, so the file stays in the current folder.")


Saved cleaned data: (15240, 13)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Summary of notebook 01

- Loaded the CarDekho data with Pandas and explored its structure, types, missing values, and duplicates.
- Found no missing values, but fixed inconsistent brand spelling, removed duplicate rows, and removed impossible or unrealistic values (0 seats, over 1,000,000 km).
- Kept the luxury cars, because their prices are real.
- Created the `km_per_year` feature.
- Built the preprocessing (standardization and one-hot encoding), fitted on training data only to avoid data leakage.
- Saved the cleaned data.

**Next:** notebook 02, exploratory data analysis with charts. It starts from `data/processed/cardekho_cleaned.csv`.